In [2]:
# open images in bball
from ultralytics import YOLO
import cv2
import numpy as np

keypoint_model = YOLO("keypoint_weights.pt")
keypoint_model2 = YOLO("keypoint2_weights.pt")


In [6]:
img = cv2.imread("/Users/davidwu/Documents/[2] Projects/Zephyr_Basketball/test_image4.jpg")
result = keypoint_model.predict(conf=0.8, source="/Users/davidwu/Documents/[2] Projects/Zephyr_Basketball/test_image3.jpg")
result2 = keypoint_model2.predict(conf=0.8, source="/Users/davidwu/Documents/[2] Projects/Zephyr_Basketball/test_image3.jpg")

results = keypoint_model2(img)[0]


image 1/1 /Users/davidwu/Documents/[2] Projects/Zephyr_Basketball/test_image3.jpg: 384x640 (no detections), 42.2ms
Speed: 1.9ms preprocess, 42.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /Users/davidwu/Documents/[2] Projects/Zephyr_Basketball/test_image3.jpg: 384x640 1 basketball-court, 1 three point line, 58.7ms
Speed: 1.2ms preprocess, 58.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 basketball-court, 1 three point line, 38.9ms
Speed: 1.4ms preprocess, 38.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)


In [5]:
def extract_court_keypoints(results):
    """Extract court corners and three-point line keypoints from segmentation masks."""
    if results.masks is None:
        return None, None
    
    masks = results.masks.data.cpu().numpy()
    class_ids = results.boxes.cls.cpu().numpy().astype(int)
    class_names = results.names
    
    court_corners = None
    three_point_keypoints = None
    
    for i, mask in enumerate(masks):
        class_name = class_names[class_ids[i]]
        
        # Process mask
        mask_binary = (mask > 0.5).astype(np.uint8) * 255
        
        if class_name == "basketball-court":
            # Find court corners
            contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                court_contour = max(contours, key=cv2.contourArea)
                
                # Approximate polygon to get corners
                epsilon = 0.02 * cv2.arcLength(court_contour, True)
                approx = cv2.approxPolyDP(court_contour, epsilon, True)
                
                # If we don't get exactly 4 points, use minimum area rectangle
                if len(approx) != 4:
                    rect = cv2.minAreaRect(court_contour)
                    corners = cv2.boxPoints(rect).astype(int)
                else:
                    corners = approx.reshape(-1, 2)
                
                # Sort corners (top-left, top-right, bottom-right, bottom-left)
                # Sort by y first to separate top and bottom
                corners = sorted(corners, key=lambda p: p[1])
                top_pts = sorted(corners[:2], key=lambda p: p[0])  # Sort by x
                bottom_pts = sorted(corners[2:], key=lambda p: p[0])  # Sort by x
                court_corners = np.array([top_pts[0], top_pts[1], bottom_pts[1], bottom_pts[0]])
                
        elif class_name == "three point line":
            # Find three point line keypoints
            contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            if contours:
                contour = max(contours, key=cv2.contourArea)
                points = contour.reshape(-1, 2)
                
                # Find the endpoints (where arc meets baseline)
                # Sort points by y-coordinate (highest y value = bottom of image)
                bottom_points = sorted(points, key=lambda p: p[1], reverse=True)[:20]
                left_end = min(bottom_points, key=lambda p: p[0])
                right_end = max(bottom_points, key=lambda p: p[0])
                
                # Find top of arc (lowest y value = top of image)
                top_of_arc = min(points, key=lambda p: p[1])
                
                three_point_keypoints = {
                    "left_end": left_end,
                    "right_end": right_end,
                    "top_of_arc": top_of_arc
                }
    
    return court_corners, three_point_keypoints

In [10]:
def visualize_masks(results, original_img):
    """
    Visualize segmentation masks from YOLO model results using only OpenCV.
    
    Args:
        results: YOLO model prediction results
        original_img: Original image to overlay masks on
    """
    if results.masks is None:
        print("No masks detected in the results")
        return
    
    # Create a copy of the original image for overlay
    overlay_img = original_img.copy()
    
    # Get image dimensions
    img_height, img_width = original_img.shape[:2]
    
    # Get masks, class IDs and class names
    masks = results.masks.data.cpu().numpy()
    class_ids = results.boxes.cls.cpu().numpy().astype(int)
    class_names = results.names
    
    # Create a blank RGB image for all masks combined
    all_masks = np.zeros_like(original_img)
    
    # Define colors for each class (can add more if needed)
    colors = {
        "basketball-court": (0, 0, 255),    # Red (BGR format)
        "three point line": (0, 255, 0),    # Green
        "baseline": (255, 0, 0),            # Blue
        "sideline": (255, 255, 0),          # Cyan
        "free throw line": (255, 0, 255),   # Magenta
    }
    
    # Create individual mask visualizations and overlay
    for i, mask in enumerate(masks):
        class_name = class_names[class_ids[i]]
        
        # Convert mask to binary and resize it to match the original image dimensions
        mask_binary = (mask > 0.5).astype(np.uint8)
        # Resize the mask to match the original image dimensions
        mask_binary_resized = cv2.resize(mask_binary, (img_width, img_height))
        
        # Get color for this class (default to white if not specified)
        color = colors.get(class_name, (255, 255, 255))
        
        # Create colored mask for this class
        colored_mask = np.zeros_like(original_img)
        colored_mask[mask_binary_resized > 0] = color
        
        # Add to the combined mask visualization
        all_masks = cv2.addWeighted(all_masks, 1, colored_mask, 0.7, 0)
        
        # Overlay on the original image with transparency
        overlay_img = cv2.addWeighted(overlay_img, 1, colored_mask, 0.5, 0)
        
        # Save individual mask for viewing
        individual_mask = np.zeros_like(original_img)
        individual_mask[mask_binary_resized > 0] = (255, 255, 255)
        cv2.imwrite(f'mask_{class_name}.png', individual_mask)
    
    # Resize all images to the same height if they have different dimensions
    height = original_img.shape[0]
    width = original_img.shape[1]
    
    # Add titles to images
    original_with_title = np.zeros((height + 40, width, 3), dtype=np.uint8)
    original_with_title[40:, :, :] = original_img
    cv2.putText(original_with_title, "Original Image", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    masks_with_title = np.zeros((height + 40, width, 3), dtype=np.uint8)
    masks_with_title[40:, :, :] = all_masks
    cv2.putText(masks_with_title, "All Masks", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    overlay_with_title = np.zeros((height + 40, width, 3), dtype=np.uint8)
    overlay_with_title[40:, :, :] = overlay_img
    cv2.putText(overlay_with_title, "Overlay", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Add a simple legend to the right side
    legend_width = 200
    legend_height = height + 40
    legend = np.ones((legend_height, legend_width, 3), dtype=np.uint8) * 0  # Black background
    
    # Add class colors to legend
    y_offset = 40
    for j, (class_name, color) in enumerate(colors.items()):
        if class_name in [class_names[cls_id] for cls_id in class_ids]:
            cv2.rectangle(legend, (10, y_offset), (30, y_offset + 20), color, -1)
            cv2.putText(legend, class_name, (40, y_offset + 15), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            y_offset += 30
    
    # Combine images horizontally
    row = cv2.hconcat([original_with_title, masks_with_title, overlay_with_title, legend])
    
    # Save the combined visualization
    cv2.imwrite('masks_visualization.png', row)
    
    # Save the overlay image separately
    cv2.imwrite('masks_overlay.png', overlay_img)
    
    return overlay_img

In [12]:

court_corners, three_point_keypoints = extract_court_keypoints(results)

# Visualize the keypoints
if court_corners is not None:
    # Draw court corners (red)
    for corner in court_corners:
        cv2.circle(img, tuple(corner), 5, (0, 0, 255), -1)
        
if three_point_keypoints is not None:
    # Draw three point keypoints (green)
    for point_name, point in three_point_keypoints.items():
        cv2.circle(img, tuple(point), 5, (0, 255, 0), -1)
        # Optional: add labels
        cv2.putText(img, point_name, tuple(point), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

cv2.imwrite('court_keypoints.png', img)
print(f"Court corners: {court_corners}")
print(f"Three-point line keypoints: {three_point_keypoints}")

Court corners: [[  2 156]
 [370 138]
 [548 323]
 [ 13 373]]
Three-point line keypoints: {'left_end': array([227, 325], dtype=int32), 'right_end': array([246, 325], dtype=int32), 'top_of_arc': array([306, 146], dtype=int32)}


In [11]:
overlay_img = visualize_masks(results, img)

# Basketball Court